### **1. Word based**

In [2]:
text = "Don't you love 🤗 Transformers? We sure do."

In [3]:
texts = text.split()
print(texts)

["Don't", 'you', 'love', '🤗', 'Transformers?', 'We', 'sure', 'do.']


In [4]:
# Split words and punctuation seperate
import re

tokens = re.split(r'(\W+)', text)
tokens = [i for i in tokens if i.strip()]
print(tokens)

['Don', "'", 't', 'you', 'love', ' 🤗 ', 'Transformers', '? ', 'We', 'sure', 'do', '.']


In [5]:
import nltk
from nltk.tokenize import word_tokenize
# nltk.download('punkt')

tokens = word_tokenize(text)
print(tokens)

['Do', "n't", 'you', 'love', '🤗', 'Transformers', '?', 'We', 'sure', 'do', '.']


### **2. Character based**

In [6]:
text = "Don't you love 🤗 Transformers? We sure do."

chars = re.split("", text.strip())
print(chars)

['', 'D', 'o', 'n', "'", 't', ' ', 'y', 'o', 'u', ' ', 'l', 'o', 'v', 'e', ' ', '🤗', ' ', 'T', 'r', 'a', 'n', 's', 'f', 'o', 'r', 'm', 'e', 'r', 's', '?', ' ', 'W', 'e', ' ', 's', 'u', 'r', 'e', ' ', 'd', 'o', '.', '']


In [7]:
# We can directly store it in list
chars = list(text)
print(chars)

['D', 'o', 'n', "'", 't', ' ', 'y', 'o', 'u', ' ', 'l', 'o', 'v', 'e', ' ', '🤗', ' ', 'T', 'r', 'a', 'n', 's', 'f', 'o', 'r', 'm', 'e', 'r', 's', '?', ' ', 'W', 'e', ' ', 's', 'u', 'r', 'e', ' ', 'd', 'o', '.']


### **3. Subword based (BPE)**

In [24]:
import re, collections

def get_stats(vocab):
    pairs = collections.defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

In [30]:
def merge_vocab(pair, v_in):
    v_out = {}
    bigram = re.escape(" ".join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in v_in:
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

In [31]:
# vocab = {'low</w>': 5, 'lower</w>': 2, 'newest</w>':6, 'widest</w>': 3}
vocab = {
    'l o w </w>': 5,
    'l o w e r </w>': 2,
    'n e w e s t </w>': 6,
    'w i d e s t </w>': 3
}

In [32]:
num_merges = 10

for i in range(num_merges):
    pairs = get_stats(vocab)
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    print(best)

('e', 's')
('es', 't')
('est', '</w>')
('l', 'o')
('lo', 'w')
('n', 'e')
('ne', 'w')
('new', 'est</w>')
('low', '</w>')
('w', 'i')


#### Naive Implementation

In [1]:
import time
import regex

PRE_TOKEN_REGEX = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

def train_bpe_tokenizer(input_path, vocab_size, special_tokens):
    # Read the corpus
    start_time = time.time()
    with open(input_path, encoding="utf-8") as f:
        corpus = f.read()

    # Initialize vocabulary with all byte values
    vocab = {idx: bytes([idx]) for idx in range(256)}

    # Pretokenize: split corpus into "words" using regex
    word_frequencies = {}
    for match in regex.finditer(PRE_TOKEN_REGEX, corpus):
        word = match.group()
        word_bytes = tuple(word.encode("utf-8"))
        word_frequencies[word_bytes] = word_frequencies.get(word_bytes, 0) + 1

    # Add special tokens to vocabulary
    for i, token in enumerate(special_tokens):
        vocab[256 + i] = token.encode("utf-8")

    merges = []
    num_merges = vocab_size - len(vocab)

    for _ in range(num_merges):
        # Count all adjacent pairs across all words
        pair_frequencies = {}
        for word, freq in word_frequencies.items():
            for i in range(len(word) - 1):
                pair = (word[i], word[i + 1])
                pair_frequencies[pair] = pair_frequencies.get(pair, 0) + freq

        if not pair_frequencies:
            break

        # Find the most frequent pair (tie-break lexicographically)
        best_pair = max(
            pair_frequencies.keys(),
            key=lambda p: (pair_frequencies[p], vocab[p[0]], vocab[p[1]])
        )

        # Create new token
        new_id = len(vocab)
        vocab[new_id] = vocab[best_pair[0]] + vocab[best_pair[1]]
        merges.append((vocab[best_pair[0]], vocab[best_pair[1]]))

        # Apply merge to all words
        new_word_frequencies = {}
        for word, freq in word_frequencies.items():
            new_word = []
            i = 0
            while i < len(word):
                if i < len(word) - 1 and (word[i], word[i + 1]) == best_pair:
                    new_word.append(new_id)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_word_frequencies[tuple(new_word)] = freq
        word_frequencies = new_word_frequencies

    end_time = time.time()
    time_completed = end_time - start_time
    print(f"Training completed in {time_completed:.2f} seconds")
    return vocab, merges, time_completed

In [2]:
vocab, merges, time_completed = train_bpe_tokenizer("input.txt", vocab_size=1000, special_tokens=["<|endoftext|>"])

Training completed in 2.53 seconds


### Version 2: Incremental Pair Updates

In [11]:
import time
import regex
from collections import Counter

In [12]:
PRE_TOKEN_REGEX = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

def get_pairs(word):
    """Return list of adjacent pairs from a word tuple of token ids (ints)."""
    return [(word[i], word[i + 1]) for i in range(len(word) - 1)]

def merge_word(word_tuple, best_pair, new_id):
    """Return a new tuple where occurrences of best_pair are replaced by new_id."""
    a, b = best_pair
    new = []
    i = 0
    while i < len(word_tuple):
        if i < len(word_tuple) - 1 and (word_tuple[i], word_tuple[i + 1]) == (a, b):
            new.append(new_id)
            i += 2
        else:
            new.append(word_tuple[i])
            i += 1
    return tuple(new)

In [13]:
def train_bpe_tokenizer_v2(input_path, vocab_size, special_tokens=(), progress=False):
    """
    Train a BPE tokenizer using incremental pair-frequency updates.
    """
    start_time = time.time()

    # Read corpus
    with open(input_path, encoding="utf-8") as f:
        corpus = f.read()

    # Initialize vocabulary with all single-byte tokens (0..255)
    vocab = {i: bytes([i]) for i in range(256)}

    # Pretokenize: split corpus into "words" using regex and count frequencies
    word_frequencies = {}
    for match in regex.finditer(PRE_TOKEN_REGEX, corpus):
        token_str = match.group()
        token_bytes = tuple(token_str.encode("utf-8"))  # sequence of ints 0..255
        word_frequencies[token_bytes] = word_frequencies.get(token_bytes, 0) + 1

    # Add special tokens to vocabulary (assigned ids starting at 256)
    for i, t in enumerate(special_tokens):
        vocab[256 + i] = t.encode("utf-8")

    merges = []
    num_merges = vocab_size - len(vocab)
    if num_merges <= 0:
        end_time = time.time()
        return vocab, merges, end_time - start_time

    # Build initial pair frequency cache once
    pair_frequencies = Counter()
    for word, freq in word_frequencies.items():
        for pair in get_pairs(word):
            pair_frequencies[pair] += freq

    # Main loop: incremental updates to pair_frequencies
    for merge_index in range(num_merges):
        if not pair_frequencies:
            break

        # Pick best pair: highest frequency, tiebreaker uses vocab bytes for determinism
        best_pair = max(
            pair_frequencies.keys(),
            key=lambda p: (pair_frequencies[p], vocab.get(p[0], b''), vocab.get(p[1], b''))
        )

        # Create new token id and bytes concatenation
        new_id = len(vocab)
        vocab[new_id] = vocab[best_pair[0]] + vocab[best_pair[1]]
        merges.append((vocab[best_pair[0]], vocab[best_pair[1]]))

        # Update only words that contain the merged pair
        new_word_frequencies = {}
        # iterate snapshot because we'll mutate pair_frequencies during the loop
        for word, freq in list(word_frequencies.items()):
            if best_pair not in get_pairs(word):
                # unchanged
                new_word_frequencies[word] = new_word_frequencies.get(word, 0) + freq
                continue

            # subtract old pair counts contributed by this word
            for pair in get_pairs(word):
                if pair in pair_frequencies:
                    pair_frequencies[pair] -= freq
                    if pair_frequencies[pair] <= 0:
                        del pair_frequencies[pair]

            # apply merge to produce merged_word
            merged_word = merge_word(word, best_pair, new_id)
            new_word_frequencies[merged_word] = new_word_frequencies.get(merged_word, 0) + freq

            # add new pair counts introduced by merged_word
            for pair in get_pairs(merged_word):
                pair_frequencies[pair] = pair_frequencies.get(pair, 0) + freq

        word_frequencies = new_word_frequencies

        if progress and (merge_index + 1) % 100 == 0:
            print(f"Completed {merge_index + 1}/{num_merges} merges; vocab size {len(vocab)}")

    end_time = time.time()
    return vocab, merges, end_time - start_time

In [14]:
vocab, merges, time_completed = train_bpe_tokenizer_v2("input.txt", vocab_size=1000, special_tokens=["<|endoftext|>"])

In [15]:
time_completed

1.2158639430999756

### Version 3: Parallel Pretokenization

In [1]:
def get_pairs(word):
    """Return list of adjacent pairs from a word tuple of token ids (ints)."""
    return [(word[i], word[i + 1]) for i in range(len(word) - 1)]

In [11]:
import os
import time
from multiprocessing import Pool

def pretokenize_chunk(chunk, special_tokens):
    """Process one chunk - runs in parallel."""
    word_frequencies = {}
    for match in regex.finditer(PRE_TOKEN_REGEX, chunk):
        word = match.group()
        word_bytes = tuple(word.encode("utf-8"))
        word_frequencies[word_bytes] = word_frequencies.get(word_bytes, 0) + 1
    return word_frequencies

# Split corpus into chunks at document boundaries
def find_chunk_boundaries(file, num_chunks, boundary_token):
    file_size = file.seek(0, os.SEEK_END)
    chunk_size = file_size // num_chunks
    boundaries = [i * chunk_size for i in range(num_chunks + 1)]
    
    # Adjust boundaries to land on special tokens (document boundaries)
    for i in range(1, len(boundaries) - 1):
        file.seek(boundaries[i])
        # Search for next occurrence of boundary token
        while True:
            chunk = file.read(4096)
            pos = chunk.find(boundary_token)
            if pos != -1:
                boundaries[i] += pos
                break
    
    return boundaries

In [5]:
def train_bpe_tokenizer_v3(input_path, vocab_size, special_tokens, num_processes=None):
    start_time = time.time()

    if num_processes is None:
        num_processes = os.cpu_count()

    # Split corpus into chunks
    with open(input_path, "rb") as f:
        boundaries = find_chunk_boundaries(f, num_processes * 3, b"<|endoftext|>")
        chunks = []
        for start, end in zip(boundaries[:-1], boundaries[1:]):
            f.seek(start)
            chunks.append(f.read(end - start).decode("utf-8"))

    # Parallel pretokenization
    with Pool(num_processes) as pool:
        chunk_frequencies = pool.starmap(
            pretokenize_chunk,
            [(chunk, special_tokens) for chunk in chunks]
        )

    # Combine word frequencies from all chunks
    word_frequencies = {}
    for chunk_freq in chunk_frequencies:
        for word, freq in chunk_freq.items():
            word_frequencies[word] = word_frequencies.get(word, 0) + freq

    # Initialize vocab with single-byte tokens
    vocab = {i: bytes([i]) for i in range(256)}

    # Add special tokens
    for i, token in enumerate(special_tokens):
        vocab[256 + i] = token.encode("utf-8")

    merges = []
    num_merges = vocab_size - len(vocab)

    # Build initial pair frequency cache once
    pair_frequencies = Counter()
    for word, freq in word_frequencies.items():
        for pair in get_pairs(word):
            pair_frequencies[pair] += freq

    # Merge loop (same as V2)
    for _ in range(num_merges):
        if not pair_frequencies:
            break

        # Find best pair
        best_pair = max(
            pair_frequencies.keys(),
            key=lambda p: (pair_frequencies[p], vocab[p[0]], vocab[p[1]])
        )

        new_id = len(vocab)
        vocab[new_id] = vocab[best_pair[0]] + vocab[best_pair[1]]
        merges.append((vocab[best_pair[0]], vocab[best_pair[1]]))

        # Only update words that contain the merged pair
        new_word_frequencies = {}

        for word, frequency in word_frequencies.items():

            if best_pair not in get_pairs(word):
                new_word_frequencies[word] = frequency
                continue

            # Remove old pair counts
            for pair in get_pairs(word):
                pair_frequencies[pair] -= frequency
                if pair_frequencies[pair] == 0:
                    del pair_frequencies[pair]

            # Apply merge
            new_word = merge_word(word, best_pair, new_id)
            new_word_frequencies[new_word] = frequency

            # Add new pair counts
            for pair in get_pairs(new_word):
                pair_frequencies[pair] = pair_frequencies.get(pair, 0) + frequency

        word_frequencies = new_word_frequencies

    end_time = time.time()
    time_completed = end_time - start_time

    print(f"Training completed in {time_completed:.2f} seconds")

    return vocab, merges, time_completed

In [6]:
vocab, merges, training_time = train_bpe_tokenizer_v3("input.txt", vocab_size=1000, special_tokens=["<|endoftext|>"])

KeyboardInterrupt: 

### Version 4: Inverted Index

In [6]:
import heapq
import json
import os
import re
from collections import defaultdict
from multiprocessing import Pool
import time
import regex
from tqdm import tqdm

PRE_TOKEN_REGEX = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

In [7]:
def _pretokenize_chunk_wrapper(args):
    """Wrapper to unpack arguments for multiprocessing.imap."""
    chunk, special_tokens = args
    return pretokenize_chunk(chunk, special_tokens)

In [8]:
# def train_bpe_tokenizer_4(input_path: str, vocab_size: int, special_tokens: list[str], num_processes: int | None = None,
# ) -> tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:
#     """
#     Train a BPE tokenizer on the input data.

#     This implementation builds on `train_bpe_tokenizer_3` by adding:
#     - Inverted index (pair → word_ids) to skip unaffected words during merges

#     Still uses max() scan for finding the best pair (O(P) per merge).

#     Args:
#         input_path: Path to the input corpus file.
#         vocab_size: Target vocabulary size.
#         special_tokens: List of special tokens to preserve.
#         num_processes: Number of parallel processes. Defaults to CPU count.

#     Returns:
#         vocab: Mapping from token ID to bytes.
#         merges: Ordered list of BPE merges.
#     """
#     start_time = time.time()

#     if num_processes is None:
#         num_processes = os.cpu_count() or 4

#     num_chunks = num_processes * 3

#     # Parallel pretokenization (same as v3)
#     chunks: list[str] = []

#     with open(input_path, "rb") as f:
#         boundary_token = special_tokens[0].encode("utf-8") if special_tokens else b"\n"
#         chunk_boundaries = find_chunk_boundaries(f, num_chunks, boundary_token)

#         for start, end in zip(chunk_boundaries[:-1], chunk_boundaries[1:]):
#             f.seek(start)
#             chunk: str = f.read(end - start).decode("utf-8", errors="ignore")
#             chunks.append(chunk)

#     with Pool(num_processes) as pool:
#         args = [(chunk, special_tokens) for chunk in chunks]
#         chunk_frequencies = list(
#             tqdm(
#                 pool.imap(_pretokenize_chunk_wrapper, args, chunksize=1),
#                 total=len(chunks),
#                 desc="Pretokenizing chunks",
#             )
#         )

#     word_frequencies_raw: dict[tuple[int, ...], int] = {}
#     for chunk_frequency in chunk_frequencies:
#         for word, frequency in chunk_frequency.items():
#             word_frequencies_raw[word] = word_frequencies_raw.get(word, 0) + frequency

#     # Above this is the same as v3

#     vocab = {idx: bytes([idx]) for idx in range(256)}

#     words: dict[int, list[int]] = {}  # word_id -> mutable token list
#     word_freqs: dict[int, int] = {}  # word_id -> frequency
#     word_id = 0
#     for word_tuple, freq in word_frequencies_raw.items():
#         words[word_id] = list(word_tuple)
#         word_freqs[word_id] = freq
#         word_id += 1

#     # Add special tokens to vocab
#     special_token_id = len(vocab)
#     for special_token in special_tokens:
#         vocab[special_token_id] = special_token.encode("utf-8")
#         special_token_id += 1

#     merges: list[tuple[bytes, bytes]] = []
#     num_merges = vocab_size - len(vocab)

#     # Build pair frequencies and inverted index
#     pair_frequencies: dict[tuple[int, int], int] = defaultdict(int)
#     pair_to_words: dict[tuple[int, int], set[int]] = defaultdict(set)

#     for wid, tokens in words.items():
#         freq = word_freqs[wid]
#         for i in range(len(tokens) - 1):
#             pair = (tokens[i], tokens[i + 1])
#             pair_frequencies[pair] += freq
#             pair_to_words[pair].add(wid)

#     # Merge loop: uses inverted index for updates
#     for _ in tqdm(range(num_merges), desc="Computing merges", unit="merge"):
#         if not pair_frequencies:
#             break

#         best_pair = max(
#             pair_frequencies.keys(), key=lambda pair: (pair_frequencies[pair], vocab[pair[0]], vocab[pair[1]])
#         )

#         new_id = len(vocab)
#         vocab[new_id] = vocab[best_pair[0]] + vocab[best_pair[1]]
#         merges.append((vocab[best_pair[0]], vocab[best_pair[1]]))

#         # Only update affected words via inverted index
#         affected_word_ids = list(pair_to_words.get(best_pair, set()))

#         for wid in affected_word_ids:
#             tokens = words[wid]
#             freq = word_freqs[wid]

#             # Remove old pair counts from frequencies and index
#             for i in range(len(tokens) - 1):
#                 p = (tokens[i], tokens[i + 1])
#                 pair_frequencies[p] -= freq
#                 if pair_frequencies[p] <= 0:
#                     del pair_frequencies[p]
#                 pair_to_words[p].discard(wid)

#             new_tokens = []
#             i = 0
#             while i < len(tokens):
#                 if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == best_pair:
#                     new_tokens.append(new_id)
#                     i += 2
#                 else:
#                     new_tokens.append(tokens[i])
#                     i += 1

#             words[wid] = new_tokens

#             # Add new pair counts to frequencies and index
#             for i in range(len(new_tokens) - 1):
#                 p = (new_tokens[i], new_tokens[i + 1])
#                 pair_frequencies[p] += freq
#                 pair_to_words[p].add(wid)

#         # Clean up the merged pair from index
#         if best_pair in pair_to_words:
#             del pair_to_words[best_pair]

#     end_time = time.time()
#     time_completed = end_time - start_time
#     print(f"Training completed in {time_completed:.2f} seconds")

#     return vocab, merges, time_completed

In [9]:
def train_bpe_tokenizer_4(input_path: str, vocab_size: int, special_tokens: list[str], num_processes: int | None = None,
) -> tuple[dict[int, bytes], list[tuple[bytes, bytes]], float]:

    start_time = time.time()

    # Read entire corpus (binary -> decode safely)
    with open(input_path, "rb") as f:
        raw = f.read()
    corpus = raw.decode("utf-8", errors="ignore")

    # Single-thread pretokenization using your existing function (unchanged)
    # pretokenize_chunk returns: dict[tuple(int...), int]
    word_frequencies_raw = pretokenize_chunk(corpus, special_tokens)

    # Build initial vocab (single-byte tokens)
    vocab = {idx: bytes([idx]) for idx in range(256)}

    # Convert word_frequencies_raw into mutable words + frequencies maps
    words: dict[int, list[int]] = {}  # word_id -> list of token ints
    word_freqs: dict[int, int] = {}  # word_id -> frequency
    word_id = 0
    for word_tuple, freq in word_frequencies_raw.items():
        words[word_id] = list(word_tuple)
        word_freqs[word_id] = freq
        word_id += 1

    # Add special tokens to vocab (preserve order)
    special_token_id = len(vocab)
    for special_token in special_tokens:
        vocab[special_token_id] = special_token.encode("utf-8")
        special_token_id += 1

    merges: list[tuple[bytes, bytes]] = []
    num_merges = vocab_size - len(vocab)
    if num_merges <= 0:
        time_completed = time.time() - start_time
        print(f"Training completed in {time_completed:.2f} seconds")
        return vocab, merges, time_completed

    # Build initial pair frequencies and inverted index: pair -> set(word_ids)
    pair_frequencies: dict[tuple[int, int], int] = defaultdict(int)
    pair_to_words: dict[tuple[int, int], set[int]] = defaultdict(set)

    for wid, tokens in words.items():
        freq = word_freqs[wid]
        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])
            pair_frequencies[pair] += freq
            pair_to_words[pair].add(wid)

    # Merge loop: use inverted index to update only affected words
    merge_iter = range(num_merges)
    if tqdm:
        merge_iter = tqdm(merge_iter, desc="Computing merges", unit="merge")
    for _ in merge_iter:
        if not pair_frequencies:
            break

        # Find best pair (tie-break by bytes stored in vocab for determinism)
        best_pair = max(
            pair_frequencies.keys(),
            key=lambda pair: (pair_frequencies[pair], vocab[pair[0]], vocab[pair[1]])
        )

        # Create new token id and record merge
        new_id = len(vocab)
        vocab[new_id] = vocab[best_pair[0]] + vocab[best_pair[1]]
        merges.append((vocab[best_pair[0]], vocab[best_pair[1]]))

        # Get affected words and update them
        affected_word_ids = list(pair_to_words.get(best_pair, set()))
        for wid in affected_word_ids:
            tokens = words[wid]
            freq = word_freqs[wid]

            # Remove old pair counts and clean inverted index entries for this word
            for i in range(len(tokens) - 1):
                p = (tokens[i], tokens[i + 1])
                pair_frequencies[p] -= freq
                if pair_frequencies[p] <= 0:
                    # safe delete
                    del pair_frequencies[p]
                # remove word id from index set (if present)
                if wid in pair_to_words.get(p, ()):
                    pair_to_words[p].discard(wid)

            # Apply the merge to tokens (replace occurrences of best_pair with new_id)
            new_tokens = []
            i = 0
            while i < len(tokens):
                if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == best_pair:
                    new_tokens.append(new_id)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1

            words[wid] = new_tokens

            # Add new pair counts and update inverted index
            for i in range(len(new_tokens) - 1):
                p = (new_tokens[i], new_tokens[i + 1])
                pair_frequencies[p] += freq
                pair_to_words[p].add(wid)

        # Remove best_pair from inverted index entirely (it no longer exists as-is)
        if best_pair in pair_to_words:
            del pair_to_words[best_pair]

    end_time = time.time()
    time_completed = end_time - start_time
    print(f"Training completed in {time_completed:.2f} seconds")

    return vocab, merges, time_completed

In [12]:
vocab, merges, training_time = train_bpe_tokenizer_4("input.txt", vocab_size=1000, special_tokens=["<|endoftext|>"])
print(f"Total training time: {training_time:.2f} seconds")
print(f"Vocabulary size: {len(vocab)}")
print(f"Number of merges: {len(merges)}")

Computing merges: 100%|██████████| 743/743 [00:00<00:00, 2276.51merge/s]

Training completed in 0.36 seconds
Total training time: 0.36 seconds
Vocabulary size: 1000
Number of merges: 743


In [13]:
vocab, merges, training_time = train_bpe_tokenizer_4("medical_corpus.txt", vocab_size=1000, special_tokens=["<|endoftext|>"])
print(f"Total training time: {training_time:.2f} seconds")
print(f"Vocabulary size: {len(vocab)}")
print(f"Number of merges: {len(merges)}")

Computing merges: 100%|██████████| 743/743 [00:08<00:00, 83.77merge/s] 


Training completed in 13.63 seconds
Total training time: 13.63 seconds
Vocabulary size: 1000
Number of merges: 743


### Version 5: Heap for Best Pair

In [14]:
def train_bpe_tokenizer_5(
    input_path: str,
    vocab_size: int,
    special_tokens: list[str],
    num_processes: int | None = None,
) -> tuple[dict[int, bytes], list[tuple[bytes, bytes]], float]:
    """
    Single-process BPE trainer with:
      - inverted index (pair -> set[word_ids]) to touch only affected words
      - heap (lazy) for O(log P) best-pair selection

    Parallel pretokenization/chunking removed (single-process).
    Returns (vocab, merges, time_completed).
    """
    import time
    import heapq
    from collections import defaultdict
    try:
        from tqdm import tqdm
    except Exception:
        tqdm = None

    start_time = time.time()

    # -------------------------
    # Single-process pretokenization
    # -------------------------
    # Read whole file and run your existing pretokenize_chunk on the whole corpus.
    with open(input_path, "rb") as f:
        raw = f.read()
    corpus = raw.decode("utf-8", errors="ignore")

    # pretokenize_chunk returns dict[tuple(int,...), int]
    word_frequencies_raw = pretokenize_chunk(corpus, special_tokens)

    # -------------------------
    # Build vocab and word maps
    # -------------------------
    vocab: dict[int, bytes] = {idx: bytes([idx]) for idx in range(256)}

    words: dict[int, list[int]] = {}      # word_id -> token id list (mutable)
    word_freqs: dict[int, int] = {}       # word_id -> frequency
    word_id = 0
    for word_tuple, freq in word_frequencies_raw.items():
        words[word_id] = list(word_tuple)
        word_freqs[word_id] = freq
        word_id += 1

    # Add special tokens (preserve order)
    next_id = len(vocab)
    for st in special_tokens:
        vocab[next_id] = st.encode("utf-8")
        next_id += 1

    merges: list[tuple[bytes, bytes]] = []
    num_merges = vocab_size - len(vocab)
    if num_merges <= 0:
        elapsed = time.time() - start_time
        print(f"Training completed in {elapsed:.2f} seconds")
        return vocab, merges, elapsed

    # -------------------------
    # Build pair frequencies + inverted index
    # -------------------------
    pair_frequencies: dict[tuple[int, int], int] = defaultdict(int)
    pair_to_words: dict[tuple[int, int], set[int]] = defaultdict(set)

    for wid, tokens in words.items():
        freq = word_freqs[wid]
        for i in range(len(tokens) - 1):
            p = (tokens[i], tokens[i + 1])
            pair_frequencies[p] += freq
            pair_to_words[p].add(wid)

    # -------------------------
    # Heap helpers
    # -------------------------
    def make_heap_entry(pair: tuple[int, int]):
        """Return a tuple usable in heap: (-freq, left_bytes, right_bytes, pair)."""
        freq = pair_frequencies.get(pair, 0)
        left_b = vocab.get(pair[0], b'')
        right_b = vocab.get(pair[1], b'')
        return (-freq, left_b, right_b, pair)

    # Build initial heap from pairs
    heap = [make_heap_entry(p) for p in pair_frequencies.keys()]
    heapq.heapify(heap)

    # valid_pairs for lazy deletion
    valid_pairs = set(pair_frequencies.keys())

    # -------------------------
    # Merge loop
    # -------------------------
    merge_iter = range(num_merges)
    if tqdm:
        merge_iter = tqdm(merge_iter, desc="Computing merges", unit="merge")

    for _ in merge_iter:
        # find best valid pair via lazy-pop
        best_pair = None
        while heap:
            negfreq, left_b, right_b, cand = heapq.heappop(heap)
            if cand not in valid_pairs:
                continue
            # check frequency freshness
            cur_freq = pair_frequencies.get(cand, 0)
            if cur_freq == 0:
                # stale -> discard
                valid_pairs.discard(cand)
                continue
            if -negfreq != cur_freq or vocab.get(cand[0], b'') != left_b or vocab.get(cand[1], b'') != right_b:
                # stale entry; re-push fresh entry and skip this one
                heapq.heappush(heap, make_heap_entry(cand))
                continue
            best_pair = cand
            break

        if best_pair is None:
            # no more pairs
            break

        # Create new token id and record merge
        new_id = len(vocab)
        vocab[new_id] = vocab[best_pair[0]] + vocab[best_pair[1]]
        merges.append((vocab[best_pair[0]], vocab[best_pair[1]]))

        # Get affected words (copy) and update them
        affected_word_ids = list(pair_to_words.get(best_pair, set()))

        for wid in affected_word_ids:
            tokens = words[wid]
            freq = word_freqs[wid]

            # Remove old pair contributions for this word
            for i in range(len(tokens) - 1):
                p = (tokens[i], tokens[i + 1])
                # decrement global count
                if p in pair_frequencies:
                    pair_frequencies[p] -= freq
                    if pair_frequencies[p] <= 0:
                        # remove stale
                        del pair_frequencies[p]
                        valid_pairs.discard(p)
                # remove wid from inverted index set
                if wid in pair_to_words.get(p, ()):
                    pair_to_words[p].discard(wid)

            # Apply merge to the word tokens (replace occurrences of best_pair)
            new_tokens = []
            i = 0
            while i < len(tokens):
                if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == best_pair:
                    new_tokens.append(new_id)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1

            words[wid] = new_tokens

            # Add new pair contributions and update inverted index
            for i in range(len(new_tokens) - 1):
                p = (new_tokens[i], new_tokens[i + 1])
                prev = pair_frequencies.get(p, 0)
                pair_frequencies[p] = prev + freq
                valid_pairs.add(p)
                pair_to_words[p].add(wid)
                # push updated entry to heap for consideration
                heapq.heappush(heap, make_heap_entry(p))

        # Remove merged pair fully from inverted index & valid set
        valid_pairs.discard(best_pair)
        if best_pair in pair_to_words:
            del pair_to_words[best_pair]
        if best_pair in pair_frequencies:
            # it should not exist, but ensure removal
            del pair_frequencies[best_pair]

        # Heuristic heap compaction to remove many stale entries
        if len(heap) > 3 * len(valid_pairs):
            heap = [make_heap_entry(p) for p in valid_pairs if p in pair_frequencies]
            heapq.heapify(heap)

    elapsed = time.time() - start_time
    print(f"Training completed in {elapsed:.2f} seconds")
    return vocab, merges, elapsed

In [15]:
vocab, merges, training_time = train_bpe_tokenizer_5("medical_corpus.txt", vocab_size=1000, special_tokens=["<|endoftext|>"])
print(f"Total training time: {training_time:.2f} seconds")
print(f"Vocabulary size: {len(vocab)}")
print(f"Number of merges: {len(merges)}")

Computing merges: 100%|██████████| 743/743 [00:06<00:00, 119.03merge/s]

Training completed in 11.76 seconds
Total training time: 11.76 seconds
Vocabulary size: 1000
Number of merges: 743


In [ ]:
if __name__ == "__main__":
    import os
    import pickle
    import time
    import tracemalloc

    INPUT_PATH = "./data/TinyStoriesV2-GPT4-train.txt"
    # INPUT_PATH = "./data/owt_train.txt"
    VOCAB_SIZE = 10_000
    # VOCAB_SIZE = 32_000
    SPECIAL_TOKENS = ["<|endoftext|>"]

    def sanitize_filename(filename: str) -> str:
        # Remove directory, remove file extension, and replace non-alphanum with underscores
        base = os.path.basename(filename)
        base_no_ext = os.path.splitext(base)[0]
        return "".join([c if c.isalnum() else "_" for c in base_no_ext])

    RESULTS_DIR = "./results"
    os.makedirs(RESULTS_DIR, exist_ok=True)
    save_file_name = sanitize_filename(INPUT_PATH)

    num_processes = os.cpu_count()

    print("Training BPE tokenizer...")
    print(f"- input path: {INPUT_PATH}")
    print(f"- vocab size: {VOCAB_SIZE}")
    print(f"- special tokens: {SPECIAL_TOKENS}")
    print(f"- num processes: {num_processes}\n")

    tracemalloc.start()
    start_time = time.time()

    vocab, merges = train_bpe_tokenizer_5(
        INPUT_PATH,
        VOCAB_SIZE,
        SPECIAL_TOKENS,
    )

    end_time = time.time()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    print(f"Training time: {end_time - start_time:.2f} seconds")
    print(f"Current memory: {current / 1024 / 1024:.2f} MB")
    print(f"Peak memory: {peak / 1024 / 1024:.2f} MB")

    longest_token = max(vocab.values(), key=len)
    print(f"Longest token: {longest_token} (length: {len(longest_token)} bytes)")
    print(f"Decoded (if valid UTF-8): {longest_token.decode('utf-8', errors='replace')}")

    with open(f"{RESULTS_DIR}/{save_file_name}_vocab.pkl", "wb") as f:
        pickle.dump(vocab, f)

    with open(f"{RESULTS_DIR}/{save_file_name}_merges.pkl", "wb") as f:
        pickle.dump(merges, f)
